In [7]:
import pandas as pd
import scanpy as sc
import numpy as np
import scipy.sparse as sp
import os
import re
import seaborn as sns
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt

from datetime import datetime

def checkpoint(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}", flush=True)


In [12]:
mp_scores = pd.read_csv("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/10_integration_GBM_HIHA/10_04_nmf_macro/mp_scores.csv", index_col=0)
adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/10_integration_GBM_HIHA/10_subset_macro.h5ad")

In [9]:
BASE = "/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune"
OUT_DIR = f"{BASE}/results/10_integration_GBM_HIHA"

In [14]:
# barcode suffix mismatch -> use intersection instead of direct .loc
common_barcodes = mp_scores.index.intersection(adata.obs_names)
n_missing = adata.n_obs - len(common_barcodes)
checkpoint(f"{len(common_barcodes)} / {adata.n_obs} adata cells matched to mp_scores "
           f"({n_missing} missing)")

[2026-09-25 11:44:04] 43358 / 43358 adata cells matched to mp_scores (0 missing)


In [15]:
mp_scores.columns.tolist()

['MP1_score', 'MP2_score', 'MP3_score', 'MP4_score', 'MP5_score', 'MP6_score']

In [17]:
# align to adata.obs_names order; unmatched cells become NaN
mp_scores_aligned = mp_scores.reindex(adata.obs_names)

def mp_sort_key(name):
    m = re.search(r"(\d+)", name)
    return int(m.group(1)) if m else 0

ALL_MPS = sorted(mp_scores.columns.tolist(), key=mp_sort_key)

for mp in ALL_MPS:
    adata.obs[mp] = mp_scores_aligned[mp].values


In [18]:
score_cols = sorted(
    [c for c in adata.obs.columns if re.match(r"MP\d+_score$", c)],
    key=lambda c: int(re.search(r"\d+", c).group())
)
score_cols = [c for c in score_cols if not re.match(r"MP6_score$", c)]
print("Detected MP score columns:", score_cols)
assert len(score_cols) > 0, (
    "No MP score columns matched. Print adata.obs.columns and check the "
    "actual naming from your scoring step, then adjust the regex above."
)

def display_mp(mp_name):
    """MP1 -> Myeloid MP1 for anything user-facing (titles, labels, sheet names)."""
    return mp_name.replace("MP", "Myleoid MP")

Detected MP score columns: ['MP1_score', 'MP2_score', 'MP3_score', 'MP4_score', 'MP5_score']


In [19]:
celltype_order = list(pd.Categorical(adata.obs["decoupler_level1_celltype"]).categories)
 
label_map = {"HIHAtlas": "Healthy", "GBMap": "GBM"}
adata.obs["condition"] = adata.obs["dataset"].map(label_map)
condition_order = ["Healthy", "GBM"]
condition_palette = {"Healthy": "#69DD82", "GBM": "#CA425F"}
 
df = adata.obs[score_cols + ["decoupler_level1_celltype", "condition"]].copy()
df["decoupler_level1_celltype"] = pd.Categorical(
    df["decoupler_level1_celltype"], categories=celltype_order, ordered=True
)
df["condition"] = pd.Categorical(df["condition"], categories=condition_order, ordered=True)
 
MIN_CELLS = 10  # skip / flag groups too small for a meaningful violin
 


In [29]:
n_rows = len(score_cols)
n_cols = len(celltype_order)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), squeeze=False)


for row, score_col in enumerate(score_cols):
    mp_name = score_col.replace("_score", "")

    for col, ct in enumerate(celltype_order):
        ax = axes[row][col]
        sub = df[df["decoupler_level1_celltype"] == ct]
        counts = sub["condition"].value_counts()
        low_n = counts.reindex(condition_order).fillna(0) < MIN_CELLS
 
        
        if sub.empty or low_n.all():
            ax.set_visible(False)
            continue
 
        sns.violinplot(
            data=sub, x="condition", y=score_col, hue = "condition",
            order=condition_order, palette=condition_palette,
            ax=ax, cut=0, inner="quartile", width=0.5, legend=False,
        )
        ax.set_title(ct, fontsize=12)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=5))

        # cell type name as column header - only on the top row
        if row == 0:
            ax.set_title(ct, fontsize=12)


        # MP name as row label - only on the leftmost column
        if col == 0:
            ax.annotate(
                f"Myeloid\n{mp_name}", xy=(0, 0.5), xytext=(-ax.yaxis.labelpad - 35, 0),
                xycoords=ax.yaxis.label, textcoords="offset points",
                fontsize=14, ha="right", va="center", 
            )

        
        # annotate n per group; flag low-n groups instead of hiding them,
        # since a missing panel is easy to misread as "no cells"
        for i, cond in enumerate(condition_order):
            n = int(counts.get(cond, 0))
            label = f"n={n}" + (" (low n)" if n < MIN_CELLS else "")
            ax.text(i, -0.10, label, ha="center", va="top", fontsize=9, transform=ax.get_xaxis_transform())
 
fig.suptitle("Myeloid MP score: Healthy vs GBM, per cell type", fontsize=18)
fig.tight_layout(rect=[0.02, 0, 1, 0.97])
 
out_path = os.path.join(OUT_DIR, "10_08_mp_violin_grid.pdf")
fig.savefig(out_path)
plt.close(fig)
print(f"Saved {out_path}")

Saved /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/10_integration_GBM_HIHA/10_08_mp_violin_grid.pdf
